In [106]:
#not all of these are used so far, here just in case
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, average_precision_score, matthews_corrcoef, recall_score, f1_score
import matplotlib.pyplot as plt
#import seaborn as sns
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold, cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

### First: Min-max scaler
### I have added code to use "distance weighting" so that close points have more weight 
Is this kernel regression?
Currently choosing/optimizing k using f1, not matthews_corrcoef or average_precision (which I think is the same as PR_AUC)

k=1 optimally, so it might be overfitting? k=3 for matthews, which produced worse f1 and especially recall but better accuracy and matthews
### Possible future tasks
Optimize k by other values like f1 score

Bring the graph back in

PRIORITY: Add another round of cross-validation in the all-caps section, if that's correct

In [107]:
#Use the results from this section!
#Maybe add the optimization graph back in and/or another visualization
#Maybe take the scaling advice from Gemini (at bottom of the "Gemini model..."" document)
#currently with Min Max scaling
#Gemini thinks this scaling method might have issues with outliers, but that's probably not the main issue with the kNN fit

heartData = pd.read_csv("df_clean.csv")
X = heartData.drop(columns=["TenYearCHD"])
y = heartData["TenYearCHD"]

# STEP 1: Initial Split
#NOTE the stratification:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=947, test_size=0.2, stratify=y)

# STEP 2: Scale (Crucial: Fit on Train only to avoid leakage)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# STEP 3: Loop to find best k using Cross-Validation
best_k = 0
best_score = 0
k_values= [k for k in range(1, 40) if k % 2 != 0]
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    # This splits X_train into 10 smaller pieces internally
    score = cross_val_score(knn, X_train_scaled, y_train, cv=10, scoring='f1').mean()
    if score > best_score:
        best_score = score
        best_k = k

# STEP 4: Final Model Training

#SHOULD I ADD another round of cross-validation here?
final_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
final_model.fit(X_train_scaled, y_train)

# STEP 5: Final Evaluation (The only time we touch the Test set)
y_pred = final_model.predict(X_test_scaled)
final_mcc = matthews_corrcoef(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Best k: {best_k}")
print(f"Reported Model Performance (MCC): {final_mcc}")
print(f"Reported Accuracy: {accuracy}")
print(f"Reported Recall: {recall}")
print(f"Reported f1 score: {f1}")

print(pd.Series(y_pred).value_counts())

Best k: 1
Reported Model Performance (MCC): 0.0833402164758265
Reported Accuracy: 0.7747641509433962
Reported Recall: 0.20155038759689922
Reported f1 score: 0.2139917695473251
0    734
1    114
Name: count, dtype: int64


### Robust Scaling (optimizing for f1 also)
might be more robust to outliers?

In [111]:
# STEP 1: Initial Split
#NOTE the stratification:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=3785, test_size=0.2, stratify=y)

# STEP 2: Scale (Crucial: Fit on Train only to avoid leakage)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# STEP 3: Loop to find best k using Cross-Validation
best_k = 0
best_score = 0
k_values= [k for k in range(1, 40) if k % 2 != 0]
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    # This splits X_train into 10 smaller pieces internally
    score = cross_val_score(knn, X_train_scaled, y_train, cv=10, scoring='f1').mean()
    if score > best_score:
        best_score = score
        best_k = k

# STEP 4: Final Model Training
#SHOULD I ADD another round of cross-validation here?
final_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
final_model.fit(X_train_scaled, y_train)

# STEP 5: Final Evaluation (The only time we touch the Test set)
y_pred = final_model.predict(X_test_scaled)
final_mcc = matthews_corrcoef(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Best k: {best_k}")
print(f"Reported Model Performance (MCC): {final_mcc}")
print(f"Reported Accuracy: {accuracy}")
print(f"Reported Recall: {recall}")
print(f"Reported f1 score: {f1}")

print(pd.Series(y_pred).value_counts())

Best k: 1
Reported Model Performance (MCC): 0.08157474490233126
Reported Accuracy: 0.7735849056603774
Reported Recall: 0.20155038759689922
Reported f1 score: 0.21311475409836064
0    733
1    115
Name: count, dtype: int64


### Standard Scaling the continuous features ONLY:

In [109]:
#Still optimizing with Matthews, but using standard scaling on only the continous features:
#PROBABLY NOT WORKING CORRECTLY
#can also try robust scaler

# STEP 1: Initial Split
#NOTE the stratification:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=3932, test_size=0.2, stratify=y)

# STEP 2: Scale (Crucial: Fit on Train only to avoid leakage)
# List your continuous columns
cont_cols = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]

# Scale only those
scaler = StandardScaler()

# Copy to avoid modifying original data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit on train
scaler.fit(X_train[cont_cols])

# Transform
X_train_scaled.loc[:, cont_cols] = scaler.transform(X_train[cont_cols])
X_test_scaled.loc[:, cont_cols] = scaler.transform(X_test[cont_cols])



# STEP 3: Loop to find best k using Cross-Validation
best_k = 0
best_score = 0
k_values= [k for k in range(1, 40) if k % 2 != 0]
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    # This splits X_train into 10 smaller pieces internally
    score = cross_val_score(knn, X_train_scaled, y_train, cv=10, scoring='f1').mean()
    if score > best_score:
        best_score = score
        best_k = k

# STEP 4: Final Model Training
final_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
final_model.fit(X_train_scaled, y_train)

# STEP 5: Final Evaluation (The only time we touch the Test set)
y_pred = final_model.predict(X_test_scaled)
final_mcc = matthews_corrcoef(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Best k: {best_k}")
print(f"Reported Model Performance (MCC): {final_mcc}")
print(f"Reported Accuracy: {accuracy}")
print(f"Reported Recall: {recall}")
print(f"Reported f1 score: {f1}")

print(pd.Series(y_pred).value_counts())

C:\Users\erica\AppData\Local\Temp\ipykernel_18416\1575579933.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.65264215  0.39862821  0.16501258 ... -1.47029687 -0.18541088
 -0.30221869]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train_scaled.loc[:, cont_cols] = scaler.transform(X_train[cont_cols])
C:\Users\erica\AppData\Local\Temp\ipykernel_18416\1575579933.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.53583433 -0.88625778  0.51543603  0.2818204   0.86585948 -0.30221869
  0.51543603 -1.47029687 -1.35348905  1.09947512 -0.88625778 -1.47029687
 -0.06860306  0.74905167 -0.65264215  0.63224385 -1.11987342 -1.23668124
 -0.88625778 -0.06860306 -1.11987342 -0.30221869 -0.53583433  0.51543603
  1.80032203 -0.06860306 -1.35348905  1.33309076  0.2818204  -0.76944996
  2.15074548 -0.886257

Best k: 1
Reported Model Performance (MCC): 0.08585390644417205
Reported Accuracy: 0.7724056603773585
Reported Recall: 0.20930232558139536
Reported f1 score: 0.21862348178137653
0    730
1    118
Name: count, dtype: int64


### Optimizing for Average Precision using ??? Scaling (horrible results)

In [110]:
#Using average precision:
#Looks worse on every metric except accuracy
#Don't use!
# STEP 3: Loop to find best k using Cross-Validation
best_k = 0
best_score = 0
k_values= [k for k in range(1, 20) if k % 2 != 0]
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    # This splits X_train into 10 smaller pieces internally
    score = cross_val_score(knn, X_train_scaled, y_train, cv=10, scoring='average_precision').mean()
    if score > best_score:
        best_score = score
        best_k = k

# STEP 4: Final Model Training
final_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
final_model.fit(X_train_scaled, y_train)

# STEP 5: Final Evaluation (The only time we touch the Test set)
y_pred = final_model.predict(X_test_scaled)
final_mcc = matthews_corrcoef(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Best k: {best_k}")
print(f"Reported Model Performance (MCC): {final_mcc}")
print(f"Reported Accuracy: {accuracy}")
print(f"Reported Recall: {recall}")
print(f"Reported f1 score: {f1}")

print(pd.Series(y_pred).value_counts())

Best k: 17
Reported Model Performance (MCC): 0.026596478788650704
Reported Accuracy: 0.8431603773584906
Reported Recall: 0.015503875968992248
Reported f1 score: 0.029197080291970802
0    840
1      8
Name: count, dtype: int64
